# DSFB-DSCD results replay (zip upload + local outputs)

This notebook replays DSCD results generated by the Rust crate.

Generate local outputs (optional):

```bash
cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201
```

Then either:
- place outputs under `output-dsfb-dscd/<timestamp>/`, or
- upload a `.zip` from your local machine (Colab picker supported).

Accepted zip layouts:
- `output-dsfb-dscd/<timestamp>/...`
- `<timestamp>/...` (where the folder contains `threshold_sweep.csv`)



In [ ]:
from pathlib import Path
from typing import List, Optional
import zipfile
import io

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception:
    widgets = None
    display = None

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None


# Optional explicit overrides
RUN_DIR = None  # e.g. Path('/content/output-dsfb-dscd/20260302_000000')
OUTPUT_ROOT = None  # e.g. Path('/content/output-dsfb-dscd')
RESULTS_ZIP_PATH = None  # e.g. Path('/content/dsfb-dscd-results.zip')


def _is_run_dir(path: Path) -> bool:
    return path.is_dir() and (path / 'threshold_sweep.csv').exists()


def _unique_paths(paths: List[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out


def _default_unpack_root() -> Path:
    if Path('/content').exists():
        return Path('/content/output-dsfb-dscd')
    return Path.cwd().resolve() / 'output-dsfb-dscd'


def _candidate_output_roots() -> List[Path]:
    roots: List[Path] = []

    if OUTPUT_ROOT is not None:
        roots.append(Path(OUTPUT_ROOT).expanduser().resolve())

    cwd = Path.cwd().resolve()
    roots.extend([
        cwd / 'output-dsfb-dscd',
        *[parent / 'output-dsfb-dscd' for parent in cwd.parents],
        Path.home() / 'output-dsfb-dscd',
        Path('/content/output-dsfb-dscd'),
        Path('/content/dsfb/output-dsfb-dscd'),
        Path('/content/dsfb/crates/dsfb-dscd/output-dsfb-dscd'),
        Path('/workspace/output-dsfb-dscd'),
        Path('/workspaces/output-dsfb-dscd'),
        Path('/tmp/output-dsfb-dscd'),
    ])

    return _unique_paths([path.resolve() for path in roots])


def _find_runs_under(root: Path) -> List[Path]:
    if not root.exists():
        return []

    if _is_run_dir(root):
        return [root]

    runs = []
    for csv_path in root.rglob('threshold_sweep.csv'):
        candidate = csv_path.parent
        if _is_run_dir(candidate):
            runs.append(candidate)
    return sorted(_unique_paths(runs))


def _latest_run_from_roots(roots: List[Path]) -> Optional[Path]:
    candidates: List[Path] = []
    for root in roots:
        candidates.extend(_find_runs_under(root))

    if not candidates:
        return None
    return sorted(_unique_paths(candidates))[-1]


def _extract_zip_bytes(zip_name: str, payload: bytes, unpack_root: Path) -> Path:
    unpack_root.mkdir(parents=True, exist_ok=True)
    print(f'Unpacking {zip_name} into {unpack_root} ...')
    with zipfile.ZipFile(io.BytesIO(payload), 'r') as archive:
        archive.extractall(unpack_root)

    run_dir = _latest_run_from_roots([unpack_root])
    if run_dir is None:
        raise FileNotFoundError(
            f'No DSCD run folder found after unpacking {zip_name} into {unpack_root}. '
            'Expected threshold_sweep.csv in a run directory.'
        )
    return run_dir


def _extract_zip_path(zip_path: Path, unpack_root: Path) -> Path:
    zip_path = Path(zip_path).expanduser().resolve()
    if not zip_path.exists():
        raise FileNotFoundError(f'Zip file not found: {zip_path}')

    unpack_root.mkdir(parents=True, exist_ok=True)
    print(f'Unpacking {zip_path} into {unpack_root} ...')
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(unpack_root)

    run_dir = _latest_run_from_roots([unpack_root])
    if run_dir is None:
        raise FileNotFoundError(
            f'No DSCD run folder found after unpacking {zip_path} into {unpack_root}. '
            'Expected threshold_sweep.csv in a run directory.'
        )
    return run_dir


def upload_and_unpack_results_zip() -> Path:
    if colab_files is None:
        raise RuntimeError('Google Colab file picker is unavailable in this environment.')

    print('Pick a DSCD results .zip from your local machine...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')

    zip_name, payload = next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'):
        raise ValueError(f'Uploaded file is not a .zip: {zip_name}')

    return _extract_zip_bytes(zip_name, payload, _default_unpack_root())


def render_zip_upload_widget() -> None:
    if widgets is None or display is None:
        print('ipywidgets unavailable; use RESULTS_ZIP_PATH or upload_and_unpack_results_zip() in Colab.')
        return

    uploader = widgets.FileUpload(accept='.zip', multiple=False, description='Upload DSCD zip')
    out = widgets.Output()

    def _on_change(change):
        value = uploader.value
        if not value:
            return

        try:
            if isinstance(value, dict):
                item = next(iter(value.values()))
                name = item.get('name', 'uploaded.zip')
                content = item.get('content', b'')
            else:
                item = value[0]
                name = item.get('name', 'uploaded.zip')
                content = item.get('content', b'')

            run_dir = _extract_zip_bytes(name, bytes(content), _default_unpack_root())
            with out:
                out.clear_output()
                print(f'Ready: {run_dir}')
        except Exception as exc:
            with out:
                out.clear_output()
                print(f'Upload failed: {exc}')

    uploader.observe(_on_change, names='value')
    display(uploader, out)


resolved_run_dir: Optional[Path] = None

if RUN_DIR is not None:
    run_dir = Path(RUN_DIR).expanduser().resolve()
    if not _is_run_dir(run_dir):
        raise FileNotFoundError(
            f'Configured RUN_DIR is not a valid DSCD run folder: {run_dir}. '
            'Expected threshold_sweep.csv inside that directory.'
        )
    resolved_run_dir = run_dir
else:
    resolved_run_dir = _latest_run_from_roots(_candidate_output_roots())

if resolved_run_dir is None and RESULTS_ZIP_PATH is not None:
    resolved_run_dir = _extract_zip_path(Path(RESULTS_ZIP_PATH), _default_unpack_root())

if resolved_run_dir is None and colab_files is not None:
    print('No local DSCD run found. Launching Colab zip picker...')
    resolved_run_dir = upload_and_unpack_results_zip()

if resolved_run_dir is None:
    searched = '\n'.join(f' - {path}' for path in _candidate_output_roots())
    print('No DSCD outputs found yet.')
    print('Options:')
    print('1) Set RUN_DIR to a run folder that contains threshold_sweep.csv')
    print('2) Set RESULTS_ZIP_PATH to a local .zip path and rerun this cell')
    print('3) In Colab, call upload_and_unpack_results_zip()')
    print('Searched:\n' + searched)
    if colab_files is None:
        render_zip_upload_widget()
    raise FileNotFoundError('No DSCD output folder available for replay.')

RUN_DIR = resolved_run_dir
OUTPUT_ROOT = RUN_DIR.parent
print(f'Using OUTPUT_ROOT: {OUTPUT_ROOT}')
print(f'Using RUN_DIR: {RUN_DIR}')
RUN_DIR


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

threshold_df = pd.read_csv(RUN_DIR / 'threshold_sweep.csv')
threshold_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['tau'], threshold_df['expansion_ratio'], linewidth=2)
ax.set_title('DSCD expansion ratio vs trust threshold')
ax.set_xlabel('tau')
ax.set_ylabel('expansion_ratio')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
finite_size_path = RUN_DIR / 'finite_size_scaling.csv'
if finite_size_path.exists():
    finite_df = pd.read_csv(finite_size_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(finite_df['num_events'], finite_df['transition_width'], marker='o')
    axes[0].set_title('Transition width vs N')
    axes[0].set_xlabel('num_events')
    axes[0].set_ylabel('transition_width')
    axes[0].grid(alpha=0.25)

    axes[1].plot(finite_df['num_events'], finite_df['max_derivative'], marker='o')
    axes[1].set_title('Max derivative vs N')
    axes[1].set_xlabel('num_events')
    axes[1].set_ylabel('max_derivative')
    axes[1].grid(alpha=0.25)
    plt.show()
else:
    print('finite_size_scaling.csv not present for this run')

In [ ]:
events_path = RUN_DIR / 'graph_events.csv'
edges_path = RUN_DIR / 'graph_edges.csv'

events_df = pd.read_csv(events_path)
edges_df = pd.read_csv(edges_path)

subset_edges = edges_df.head(128)
subset_nodes = sorted(set(subset_edges['from_event_id']).union(subset_edges['to_event_id']))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[['from_event_id', 'to_event_id']].itertuples(index=False, name=None))

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(graph, seed=7)
nx.draw_networkx(graph, pos=pos, node_size=120, with_labels=False, arrows=True)
plt.title('DSCD graph preview')
plt.axis('off')
plt.show()